## Cargar datos con Web Scraping

In [1]:
# Dependencias para cargar y explorar los datos.
# %pip install pandas numpy beautifulsoup4

In [2]:
import pandas as pd
import numpy as np
from urllib.request import urlopen
from bs4 import BeautifulSoup

# Sitio de práctica: precios y calificaciones ficticios. Los precios están en GBP.
url = 'https://books.toscrape.com/'
with urlopen(url, timeout=20) as respuesta:
    soup = BeautifulSoup(respuesta.read().decode('utf-8'), 'html.parser')

# select localiza los libros; cada diccionario será una fila de la tabla.
libros = []
for libro in soup.select('article.product_pod'):
    titulo = libro.select_one('h3 a')
    precio = libro.select_one('.price_color')
    calificacion = libro.select_one('p.star-rating')
    # Omitir elementos que no tengan los tres campos necesarios.
    if titulo is None or precio is None or calificacion is None:
        continue
    libros.append({
        'titulo': titulo['title'],
        'precio_gbp': float(precio.get_text().replace('£', '')),
        'calificacion': calificacion['class'][1]
    })
df = pd.DataFrame(libros)
df['caracteres_titulo'] = df['titulo'].str.len()
# np.mean calcula el precio promedio de los libros de esta página.
print(f'Precio promedio: {np.mean(df["precio_gbp"]):.2f} GBP')
df.head()

Precio promedio: 38.05 GBP


,titulo,precio_gbp,calificacion,caracteres_titulo
0,A Light in the Attic,51.77,Three,20
1,Tipping the Velvet,53.74,One,18
2,Soumission,50.10,One,10
3,Sharp Objects,47.82,Four,13
4,Sapiens: A Brief History of Humankind,54.23,Five,37


## Manejo de Valores Faltantes y Duplicados

In [3]:
# 1. Identificar valores nulos en el dataset
# isnull detecta los vacíos y sum los cuenta por columna.
df.isnull().sum()

titulo               0
precio_gbp           0
calificacion         0
caracteres_titulo    0
dtype: int64

In [4]:
# 2. Eliminar valores nulos en el dataset
# dropna devuelve una tabla sin filas incompletas; df no se modifica.
df.dropna().head()

,titulo,precio_gbp,calificacion,caracteres_titulo
0,A Light in the Attic,51.77,Three,20
1,Tipping the Velvet,53.74,One,18
2,Soumission,50.10,One,10
3,Sharp Objects,47.82,Four,13
4,Sapiens: A Brief History of Humankind,54.23,Five,37


In [5]:
# 3. Rellenar valores nulos en el dataset
# fillna muestra el reemplazo por cero sin modificar df.
# El cero debe tener sentido para la columna que se analiza.
df.fillna(0).head()

,titulo,precio_gbp,calificacion,caracteres_titulo
0,A Light in the Attic,51.77,Three,20
1,Tipping the Velvet,53.74,One,18
2,Soumission,50.10,One,10
3,Sharp Objects,47.82,Four,13
4,Sapiens: A Brief History of Humankind,54.23,Five,37


In [6]:
# 4. Rellenar valores nulos con la media de la columna
# mean calcula la media; fillna reemplaza solo los valores ausentes.
df['precio_gbp'] = df['precio_gbp'].fillna(df['precio_gbp'].mean())

In [7]:
# 5. Verificar si hay valores duplicados en el dataset
# duplicated crea una máscara; keep=False incluye todas las apariciones.
# Repetir un valor de una columna no implica que la fila completa sea duplicada.
mascara = df.duplicated(subset=['calificacion'], keep=False)
# loc selecciona las filas; DataFrame conserva el resultado como una tabla.
duplicados = pd.DataFrame(df.loc[mascara])
duplicados.sort_values(by='calificacion').head(10)

,titulo,precio_gbp,calificacion,caracteres_titulo
4,Sapiens: A Brief History of Humankind,54.23,Five,37
12,Set Me Free,17.46,Five,11
13,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,Five,55
14,Rip it Up and Start Again,35.02,Five,25
3,Sharp Objects,47.82,Four,13
6,The Dirty Little Secrets of Getting Your Dream...,33.34,Four,50
8,The Boys in the Boat: Nine Americans and Their...,22.60,Four,94
11,Shakespeare's Sonnets,20.66,Four,21
9,The Black Maria,52.15,One,15
1,Tipping the Velvet,53.74,One,18


## 2. Codifique variables categoricas usando LabEncoder y One-Hot Encoding.

In [8]:
# Label encoding representa cada categoría mediante un código.
# One-Hot Encoding crea una columna de 0 y 1 por cada categoría.
# Son formas de representar categorías al preparar datos para un modelo.

## 3. Aplique normalizacion y estandarizacion.

In [9]:
# La normalización ajusta los valores a una escala, por ejemplo entre 0 y 1.
# La estandarización resta la media y divide por la desviación estándar.
# En un modelo, estos parámetros se calculan solo con sus datos de entrenamiento.

## 4. Balancea la variable objetivo usando SMOTE.

In [10]:
# SMOTE genera ejemplos sintéticos para las clases con menos observaciones.
# Se utiliza al preparar un clasificador y se aplica solo a sus datos de entrenamiento.

## 5. Muestre el conjunto de datos final listo para entrenar un modelo de clasificacion.

In [11]:
# Un conjunto para clasificación separa las variables de entrada de la variable objetivo.
# Su preparación depende del problema y del modelo que se vaya a utilizar.